In [1]:
import torch
import os
import sys
from datasets import load_dataset

sys.path.insert(0, "..")
from testbed.data import prepare_dataloader
import config
import exp_settings as setting

dataset = load_dataset(
    os.path.join(config.testbed_dir, "data", "vqav2"),
    split="validation",
    data_dir=".",
    images_dir=config.coco_dir,
    trust_remote_code=True,
)

hparams = {
    "batch_size": 8,
    "num_shots": 0,
    "dtype": torch.float16,
    "generate_args": setting.generate_args,
}

dataloader = prepare_dataloader(
    dataset,
    batch_size=hparams["batch_size"],
    num_shots=hparams["num_shots"],
)

In [2]:
from transformers import BitsAndBytesConfig
from testbed.models import Idefics
from dev.icv_encoder import GlobalICVEncoder

device = torch.device("cuda:1")
lmm = Idefics(
    config.idefics_9b_path,
    torch_dtype=torch.float16,
).to(device)
lmm.eval()
icv_encoder = GlobalICVEncoder(4096, 32).to(device)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [4]:
sd = torch.load("../results/ckpt/icv_cpk-0.pth")
sd = {
    k.removeprefix("icv_encoder."): v.squeeze()
    for k, v in sd.items()
    if k.startswith("icv_encoder.")
}
icv_encoder.load_state_dict(sd, strict=False)

<All keys matched successfully>

In [6]:
from testbed.models.model_base import HookType


hooks = lmm.register_forward_hook(
    HookType.TEXT_MODEL_LAYER,
    icv_encoder.hook,
)

In [8]:
for hook in hooks:
    hook.remove()

In [10]:
from tqdm import tqdm
import evaluate

from testbed.data import prepare_vqa_input
from testbed.data.vqav2 import postprocess_generation

total_acc = evaluate.load("Kamichanw/vqa_accuracy")
result = []

# for simplicity, just run 10 batches
for batch in tqdm(dataloader, desc=f"Evaluating {lmm.model_name} ..."):
    text, images = prepare_vqa_input(
        batch, instruction="Provide an answer to the question. Use the image to answer."
    )
    predictions = lmm.generate(text, images, **hparams["generate_args"])
    for pred, context in zip(predictions, batch):
        last_qa = context[-1]
        gt_answer = [item["answer"] for item in last_qa["answers"]]
        prediction = postprocess_generation(pred)
        total_acc.add(
            prediction=prediction,
            reference=gt_answer,
            question_types=last_qa["question_type"],
            answer_types=last_qa["answer_type"],
        )
        result.append(
            {
                "question_id": last_qa["question_id"],
                "raw_output": pred,
                "question": last_qa["question"],
                "question_type": last_qa["question_type"],
                "answer_type": last_qa["answer_type"],
                "prediction": prediction,
                "answers": last_qa["answers"],
            }
        )
    break

eval_result = total_acc.compute()
eval_result

Evaluating idefics-9b ...:   0%|          | 0/1250 [00:01<?, ?it/s]


{'overall': 45.0,
 'perAnswerType': {'other': 40.0, 'yes/no': 66.66666666666667, 'number': 0.0},
 'perQuestionType': {'what is the man': 0.0,
  'are there': 100.0,
  'what kind of': 100.0,
  'is the': 0.0,
  'how many': 0.0,
  'what is': 0.0,
  'do you': 100.0,
  'none of the above': 60.0}}

In [9]:
hparams["dtype"] = str(hparams["dtype"])
evaluate.save("./", eval_result=eval_result, hparams=hparams, records=result)

PosixPath('result-2024_09_14-09_34_14.json')